# 05 · huSpaCy: a Hungarian-specific encoder vs e5

**Question:** does an encoder built for Hungarian predict human picks better than the multilingual e5-small the
champion uses? The cards are Hungarian jokes, judged by native speakers: wordplay, slang and suffix-heavy words
are where a multilingual model is weakest.

**The encoder:** [huspacy/hu_core_news_md](https://huggingface.co/huspacy/hu_core_news_md) (v3.8.1, CC-BY-SA-4.0),
a spaCy pipeline with 100-dimensional *floret* word vectors (built from sub-word pieces, so unseen and suffixed
words still get a vector) and a lemmatiser. A card's vector is the average of its words' vectors, in two variants:
- **tokens**: the words as written;
- **lemmas**: the dictionary forms ("naplóm" → "napló", "padláson" → "padlás"), which strips the suffixes.

Note that this is a bag of word vectors, not a sentence model trained to compare texts like e5.

**How it's measured:** the training cycle's clean split (val/test: human picks from games without a computer
player), the pipeline's two model structures (`train.make_scorer`, `train.CONFIGS`) on each encoder, and the
difference in MRR to e5 with a 95% interval from a paired bootstrap over games.

**Extra dependency** (not part of the locked environments), installed with the lock files as constraints:
`venv-ml/bin/pip install spacy==3.8.18 "hu_core_news_md @ https://huggingface.co/huspacy/hu_core_news_md/resolve/f6cfd29ca9db9d809d095954ff209b0099343a7f/hu_core_news_md-3.8.1-py3-none-any.whl"`.
Embeddings are cached in `ml-data/embeddings-experiments/`.

In [ ]:
import os
import sys
from functools import partial
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "ml").is_dir() else Path.cwd().parent  # papermill: repo root; VS Code: notebooks/
sys.path.insert(0, str(ROOT))
os.environ.setdefault("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")

import numpy as np
import pandas as pd
import torch

from ml import evaluate, features, lower_priority, train
from ml import experiments as ex

try:
    import spacy
except ImportError as exc:
    raise SystemExit("spaCy isn't installed in this environment: see the install line above") from exc

print("CPU priority:", lower_priority())  # idle CPU only: the game comes first

In [ ]:
snapshot = None
modes = ["tokens", "lemmas"]   # how a card becomes one vector (see above)
reference = "e5"               # the champion's encoder
model_types = ["linear", "mlp"]
log_mlflow = True

In [ ]:
data = ex.load(snapshot)
print(data.snapshot.name, data.sizes())
nlp = spacy.load("hu_core_news_md")
print("huSpaCy", nlp.meta["version"], "| vector width", nlp.vocab.vectors.shape[1])

In [ ]:
class SpacyEncoder:
    """features.Encoder for huSpaCy: the average of the words' (or lemmas') vectors, length-normalised."""

    def __init__(self, nlp, mode: str):
        self.nlp, self.mode = nlp, mode

    def encode(self, texts):
        out = np.zeros((len(texts), self.nlp.vocab.vectors.shape[1]), dtype=np.float32)
        if self.mode == "tokens":
            docs = (self.nlp.make_doc(t) for t in texts)  # tokeniser only: fast
        else:
            docs = self.nlp.pipe(texts, disable=["parser", "ner"], batch_size=64)  # lemmas need the tagger
        for i, doc in enumerate(docs):
            words = [t for t in doc if not (t.is_punct or t.is_space)]
            if self.mode == "tokens":
                vecs = [t.vector for t in words]
            else:
                vecs = [self.nlp.vocab[t.lemma_.lower()].vector for t in words]
            if vecs:
                v = np.mean(vecs, axis=0)
                norm = np.linalg.norm(v)
                out[i] = v / norm if norm else v
        return out


def huspacy_model(mode: str) -> features.EmbeddingModel:
    return features.EmbeddingModel(f"huspacy-md-{mode}", "huspacy/hu_core_news_md",
                                   f"{nlp.meta['version']}-f6cfd29c", dim=nlp.vocab.vectors.shape[1])


encoders = {mode: (huspacy_model(mode), SpacyEncoder(nlp, mode)) for mode in modes}

In [ ]:
# Sanity check: the white cards closest to one black card, by cosine similarity, per encoder.
black = next(e.black for e in data.splits["test"])
whites = sorted({c for e in data.splits["test"][:40] for c in e.hand})
for mode, (_model, encoder) in encoders.items():
    vecs = encoder.encode([data.texts[black]] + [data.texts[w] for w in whites])
    top = np.argsort(-(vecs[1:] @ vecs[0]))[:3]
    print(f"[{mode}] {data.texts[black]!r} ->", [data.texts[whites[i]] for i in top])

In [ ]:
inputs, targets = {}, {}
inputs[reference], targets[reference] = data.tensors(reference)
for mode, (model, encoder) in encoders.items():
    inputs[mode], targets[mode] = data.tensors(model, encoder_factory=lambda m, enc=encoder: enc)

results = ex.baselines(data)
runs = {}  # (encoder, model type) -> Result
for enc in [reference, *modes]:
    dim = inputs[enc]["train"][0].shape[-1]
    for m in model_types:
        name = f"{m} {enc}" if enc == reference else f"{m} huspacy-{enc}"
        r = ex.fit(name, partial(train.make_scorer, dim, train.CONFIGS[m]), inputs[enc], targets[enc],
                   train.CONFIGS[m], {"model_type": m, "embedding": enc if enc == reference else f"huspacy-md-{enc}"})
        runs[(enc, m)] = r
        results.append(r)
ex.table(results)

In [ ]:
def ranks(enc, m, split):
    model = runs[(enc, m)].model
    with torch.no_grad():
        probs = torch.softmax(model(*inputs[enc][split]), dim=-1).numpy()
    return evaluate.reciprocal_ranks(probs, data.targets[split])

rows = []
for m in model_types:
    for mode in modes:
        row = {"huSpaCy variant": f"{m} · {mode}"}
        for split in ("val", "test"):
            b = evaluate.paired_bootstrap(ranks(reference, m, split), ranks(mode, m, split),
                                          [e.game_id for e in data.splits[split]])
            row[f"{split} MRR {reference}"] = runs[(reference, m)].scores[split]["mrr"]
            row[f"{split} MRR huSpaCy"] = runs[(mode, m)].scores[split]["mrr"]
            row[f"{split} Δ [95%]"] = f"{b['delta']:+.3f} [{b['lo']:+.3f}, {b['hi']:+.3f}]"
            row[f"{split} verdict"] = ("better" if b["lo"] > 0 else "worse" if b["hi"] < 0 else "no difference")
        rows.append(row)
comparison = pd.DataFrame(rows).set_index("huSpaCy variant").round(4)
comparison

In [ ]:
best = max(((mode, m) for mode in modes for m in model_types), key=lambda k: runs[k].scores["val"]["mrr"])
champ = runs[(reference, "linear")].scores
print(f"Best huSpaCy variant on validation: {best[1]} · {best[0]} "
      f"(test MRR {runs[best].scores['test']['mrr']:.3f}, top1 {runs[best].scores['test']['top1']:.3f})")
print(f"The champion's setup, linear {reference}: test MRR {champ['test']['mrr']:.3f}, top1 {champ['test']['top1']:.3f}")
print("Verdicts on test:", dict(comparison["test verdict"]))

In [ ]:
if log_mlflow:
    print("MLflow parent run:", ex.log("05_huspacy_encoder", data, results,
                                      {"modes": ",".join(modes), "reference": reference,
                                       "huspacy_version": nlp.meta["version"]}))